In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
sales = pd.read_csv("../data/raw/sales_train.csv", parse_dates=["date"])
sales.info()

<class 'pandas.DataFrame'>
RangeIndex: 4007419 entries, 0 to 4007418
Data columns (total 14 columns):
 #   Column           Dtype         
---  ------           -----         
 0   unique_id        int64         
 1   date             datetime64[us]
 2   warehouse        str           
 3   total_orders     float64       
 4   sales            float64       
 5   sell_price_main  float64       
 6   availability     float64       
 7   type_0_discount  float64       
 8   type_1_discount  float64       
 9   type_2_discount  float64       
 10  type_3_discount  float64       
 11  type_4_discount  float64       
 12  type_5_discount  float64       
 13  type_6_discount  float64       
dtypes: datetime64[us](1), float64(11), int64(1), str(1)
memory usage: 459.1 MB


In [3]:
sales.describe().T

,count,mean,min,25%,50%,75%,max,std
unique_id,4007419.0,2740.801319,0.0,1398.0,2744.0,4084.0,5431.0,1558.566407
date,4007419,2022-09-17 11:44:05.398149,2020-08-01 00:00:00,2021-11-09 00:00:00,2022-10-23 00:00:00,2023-08-17 00:00:00,2024-06-02 00:00:00,NaN
total_orders,4007367.0,5995.722112,458.0,4402.0,5515.0,8011.0,18475.0,2479.5978
sales,4007367.0,108.380753,0.0,17.98,39.52,94.47,26316.19,360.332735
sell_price_main,4007419.0,181.419909,0.02,21.13,45.64,112.13,21682.99,461.101407
availability,4007419.0,0.925311,0.01,1.0,1.0,1.0,1.0,0.179621
type_0_discount,4007419.0,0.008519,-20.9493,0.0,0.0,0.0,0.99164,0.05147
type_1_discount,4007419.0,0.000124,0.0,0.0,0.0,0.0,0.5,0.005143
type_2_discount,4007419.0,0.001163,0.0,0.0,0.0,0.0,0.76372,0.0188
type_3_discount,4007419.0,0.000005,0.0,0.0,0.0,0.0,0.25063,0.000892


In [4]:
sales[sales.duplicated()].shape

(0, 14)

In [5]:
sales.groupby(['warehouse', 'date'])['total_orders'].value_counts()

warehouse  date        total_orders
Brno_1     2020-08-01  4797.0          318
                       4991.0           33
           2020-08-02  4888.0          312
                       4919.0           33
           2020-08-03  5588.0          316
                                      ... 
Prague_3   2024-05-31  6295.0          105
           2024-06-01  5240.0          531
                       5823.0          105
           2024-06-02  5177.0          518
                       5392.0          105
Name: count, Length: 17632, dtype: int64

In [6]:
sales.groupby('warehouse').agg(
    start_date = ('date', 'min'), end_date = ('date', 'max')
)

,start_date,end_date
warehouse,,
Brno_1,2020-08-01,2024-06-02
Budapest_1,2020-08-01,2024-06-02
Frankfurt_1,2021-12-08,2024-06-02
Munich_1,2021-05-20,2024-06-02
Prague_1,2020-08-01,2024-06-02
Prague_2,2020-08-01,2024-06-02
Prague_3,2020-08-01,2024-06-02


### 📊 Sales Analysis
**Period:** `2020-08` — `2024-06`,
**Latest Warehouses:** `Frankfurt_1 - 2021-12-08 and Munich 2021-05-20`
---


In [7]:
sales.isnull().sum()

unique_id           0
date                0
warehouse           0
total_orders       52
sales              52
sell_price_main     0
availability        0
type_0_discount     0
type_1_discount     0
type_2_discount     0
type_3_discount     0
type_4_discount     0
type_5_discount     0
type_6_discount     0
dtype: int64

### ⚠️ Missing Values Assessment
Out of a total of **4,007,4197** records, missing values were found in:
* **`total_orders`**: 52 missing
* **`sales`**: 52 missing

### 🏷️ Promotional Features
**Columns:** `type_0_discount`, `type_1_discount`

These features represent different types of promotional sales run by Rohlik, indicating the discount as a percentage of the original price.

* **Calculation:** `(original_price - current_price) / original_price`
* **Application Rule:** Multiple discount types can run simultaneously, but the **highest** possible discount is always the one applied to the sale.
* **Data Note:** Negative discount values should be treated as **no discount** (0%).

In [8]:
orders_per_day = sales.groupby(['warehouse', 'date'])['total_orders'].nunique(dropna=True)
anomalies = orders_per_day[orders_per_day > 1]
print("Days when unexpectedly more than 2 unique values of total_orders:", len(anomalies))

Days when unexpectedly more than 2 unique values of total_orders: 7517


In [9]:
sales.groupby('warehouse')['sell_price_main'].mean()

warehouse
Brno_1          67.873869
Budapest_1     918.184147
Frankfurt_1      2.977863
Munich_1         3.002412
Prague_1        65.941507
Prague_2        65.624530
Prague_3        67.141527
Name: sell_price_main, dtype: float64

In [10]:
sales.groupby('warehouse')['sell_price_main'].median()

warehouse
Brno_1          44.64
Budapest_1     696.80
Frankfurt_1      2.20
Munich_1         2.18
Prague_1        43.86
Prague_2        43.66
Prague_3        44.08
Name: sell_price_main, dtype: float64

### 🏢 Main Sell Price by Warehouse

Sell Prices are in country's own currency

| Warehouse | Sell Price Main | Currency |
| :--- | :---: | :---: |
| **Brno_1** | 44.64 | CZK |
| **Budapest_1** | 696.80 | HUF |
| **Frankfurt_1** | 2.20 | EUR |
| **Munich_1** | 2.18 | EUR |
| **Prague_1** | 43.86 | CZK |
| **Prague_2** | 43.66 | CZK |
| **Prague_3** | 44.08 | CZK |

In [11]:
calendar = pd.read_csv("../data/raw/calendar.csv", parse_dates=['date'])
calendar.head()

,date,holiday_name,holiday,shops_closed,winter_school_holidays,school_holidays,warehouse
0,2022-03-16,NaN,0,0,0,0,Frankfurt_1
1,2020-03-22,NaN,0,0,0,0,Frankfurt_1
2,2018-02-07,NaN,0,0,0,0,Frankfurt_1
3,2018-08-10,NaN,0,0,0,0,Frankfurt_1
4,2017-10-26,NaN,0,0,0,0,Prague_2


In [12]:
sales_calender = pd.merge(sales, calendar, on = ['warehouse', 'date'], how = 'left')
sales_calender

,unique_id,date,warehouse,total_orders,sales,sell_price_main,availability,type_0_discount,type_1_discount,type_2_discount,type_3_discount,type_4_discount,type_5_discount,type_6_discount,holiday_name,holiday,shops_closed,winter_school_holidays,school_holidays
0,4845,2024-03-10,Budapest_1,6436.0,16.34,646.26,1.00,0.00000,0.0,0.0,0.0,0.15312,0.0,0.0,NaN,0,0,0,0
1,4845,2021-05-25,Budapest_1,4663.0,12.63,455.96,1.00,0.00000,0.0,0.0,0.0,0.15025,0.0,0.0,NaN,0,0,0,0
2,4845,2021-12-20,Budapest_1,6507.0,34.55,455.96,1.00,0.00000,0.0,0.0,0.0,0.15025,0.0,0.0,NaN,0,0,0,0
3,4845,2023-04-29,Budapest_1,5463.0,34.52,646.26,0.96,0.20024,0.0,0.0,0.0,0.15312,0.0,0.0,NaN,0,0,0,0
4,4845,2022-04-01,Budapest_1,5997.0,35.92,486.41,1.00,0.00000,0.0,0.0,0.0,0.15649,0.0,0.0,NaN,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4007414,4941,2023-06-21,Prague_1,9988.0,26.56,34.06,1.00,0.00000,0.0,0.0,0.0,0.00000,0.0,0.0,NaN,0,0,0,0
4007415,4941,2023-06-24,Prague_1,8518.0,27.42,34.06,1.00,0.00000,0.0,0.0,0.0,0.00000,0.0,0.0,NaN,0,0,0,0
4007416,4941,2023-06-23,Prague_1,10424.0,33.39,34.06,1.00,0.00000,0.0,0.0,0.0,0.00000,0.0,0.0,NaN,0,0,0,0
4007417,4941,2023-06-22,Prague_1,10342.0,22.88,34.06,1.00,0.00000,0.0,0.0,0.0,0.00000,0.0,0.0,NaN,0,0,0,0


# Missing values
From 52 missing values - 42 of them are without values unreasonably. 10 of them are missed due to holidays
The 42 missing gaps can be **removed**.

In [13]:
sales_calender.isnull().sum()

unique_id                       0
date                            0
warehouse                       0
total_orders                   52
sales                          52
sell_price_main                 0
availability                    0
type_0_discount                 0
type_1_discount                 0
type_2_discount                 0
type_3_discount                 0
type_4_discount                 0
type_5_discount                 0
type_6_discount                 0
holiday_name              3844119
holiday                         0
shops_closed                    0
winter_school_holidays          0
school_holidays                 0
dtype: int64

In [14]:
sales_calender[(sales_calender['total_orders'].isnull()) & (sales_calender['sales'].isnull()) & (sales_calender['holiday'] == 0)]

,unique_id,date,warehouse,total_orders,sales,sell_price_main,availability,type_0_discount,type_1_discount,type_2_discount,type_3_discount,type_4_discount,type_5_discount,type_6_discount,holiday_name,holiday,shops_closed,winter_school_holidays,school_holidays
154099,885,2021-05-21,Munich_1,NaN,NaN,4.02,1.00,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0,0,0,0
154155,885,2021-05-22,Munich_1,NaN,NaN,4.02,1.00,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0,0,0,0
429815,1237,2021-12-10,Frankfurt_1,NaN,NaN,3.84,0.30,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0,0,0,0
429860,1237,2021-12-09,Frankfurt_1,NaN,NaN,3.84,1.00,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0,0,0,0
733468,725,2021-06-27,Munich_1,NaN,NaN,2.82,1.00,0.10000,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0,0,0,0
734198,725,2021-06-26,Munich_1,NaN,NaN,2.82,1.00,0.10000,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0,0,0,0
738594,3778,2021-05-29,Munich_1,NaN,NaN,2.75,1.00,0.24460,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0,0,0,0
738632,3778,2021-05-31,Munich_1,NaN,NaN,2.75,0.22,0.24460,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0,0,0,0
738769,3778,2021-05-22,Munich_1,NaN,NaN,2.75,1.00,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0,0,0,0
738805,3778,2021-05-30,Munich_1,NaN,NaN,2.75,1.00,0.24460,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0,0,0,0


In [15]:
sales_calender[(sales_calender['sales'] == 0) & (sales_calender['holiday'] == 0)]

,unique_id,date,warehouse,total_orders,sales,sell_price_main,availability,type_0_discount,type_1_discount,type_2_discount,type_3_discount,type_4_discount,type_5_discount,type_6_discount,holiday_name,holiday,shops_closed,winter_school_holidays,school_holidays
152,4845,2020-10-13,Budapest_1,4144.0,0.0,494.02,1.00,0.0,0.0,0.0,0.0,0.00000,0.0,0.0,NaN,0,0,0,0
355,4845,2021-05-29,Budapest_1,4250.0,0.0,455.96,1.00,0.0,0.0,0.0,0.0,0.15025,0.0,0.0,NaN,0,0,0,0
385,4845,2020-08-09,Budapest_1,3470.0,0.0,494.02,1.00,0.0,0.0,0.0,0.0,0.00000,0.0,0.0,NaN,0,0,0,0
715,4845,2020-08-06,Budapest_1,3571.0,0.0,494.02,1.00,0.0,0.0,0.0,0.0,0.00000,0.0,0.0,NaN,0,0,0,0
826,4845,2020-12-15,Budapest_1,4794.0,0.0,494.02,1.00,0.0,0.0,0.0,0.0,0.00000,0.0,0.0,NaN,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4005936,5061,2021-07-15,Prague_2,4052.0,0.0,110.46,0.03,0.0,0.0,0.0,0.0,0.00000,0.0,0.0,NaN,0,0,0,0
4006454,1081,2023-09-26,Brno_1,8014.0,0.0,22.88,0.02,0.0,0.0,0.0,0.0,0.00000,0.0,0.0,NaN,0,0,0,0
4006646,1218,2023-06-23,Budapest_1,5916.0,0.0,1416.83,0.03,0.0,0.0,0.0,0.0,0.00000,0.0,0.0,NaN,0,0,0,0
4006647,1218,2023-06-20,Budapest_1,5641.0,0.0,1416.83,0.06,0.0,0.0,0.0,0.0,0.00000,0.0,0.0,NaN,0,0,0,0


In [16]:
raws_sales_drop = sales_calender[(sales_calender['sales'] == 0) & (sales_calender['holiday'] == 0)].shape
raws_sales_drop

(41791, 19)

41791 columns with 0 sales - maybe goods had been already sold or nobody bought them that day

In [17]:
raws_sales_drop = sales_calender[(sales_calender['total_orders'].isnull()) & (sales_calender['sales'].isnull()) & (sales_calender['holiday'] == 0)].index
raws_sales_drop

Index([ 154099,  154155,  429815,  429860,  733468,  734198,  738594,  738632,
        738769,  738805,  738816, 1028203, 1028753, 1231189, 1231500, 1231580,
       1231636, 1231755, 1451592, 1451661, 1451854, 1451892, 1452131, 1452275,
       1452400, 1452632, 3464921, 3464993, 3465126, 3465135, 3465197, 3744049,
       3860150, 3860232, 3862121, 3862202, 3940612, 3940655, 3940665, 3940702,
       4002178, 4002183],
      dtype='int64')

In [18]:
sales_calender = sales_calender.drop(index = raws_sales_drop)

In [19]:
sales_calender.groupby('warehouse')[['unique_id', 'date', 'total_orders']].nunique()

,unique_id,date,total_orders
warehouse,,,
Brno_1,757,1402,1959
Budapest_1,940,1402,1782
Frankfurt_1,483,853,558
Munich_1,640,1089,1273
Prague_1,857,1402,2051
Prague_2,853,1402,2055
Prague_3,860,1402,1820


In [20]:
sales_calender['type_0_discount'] = sales_calender['type_0_discount'].clip(lower=0).fillna(0)
sales_calender['type_1_discount'] = sales_calender['type_1_discount'].clip(lower=0).fillna(0)
sales_calender['type_2_discount'] = sales_calender['type_2_discount'].clip(lower=0).fillna(0)
sales_calender['type_3_discount'] = sales_calender['type_3_discount'].clip(lower=0).fillna(0)
sales_calender['type_4_discount'] = sales_calender['type_4_discount'].clip(lower=0).fillna(0)
sales_calender['type_5_discount'] = sales_calender['type_5_discount'].clip(lower=0).fillna(0)
sales_calender['type_6_discount'] = sales_calender['type_6_discount'].clip(lower=0).fillna(0)

In [21]:
cols = ['type_0_discount', 'type_1_discount', 'type_2_discount', 'type_3_discount', 'type_4_discount', 'type_5_discount', 'type_6_discount']

In [22]:
sales_calender['max_discount'] = sales_calender[cols].max(axis=1)
sales_calender = sales_calender.drop(columns=cols)

In [23]:
sales_calender['is_discount'] = (sales_calender['max_discount'] > 0).astype(int)

In [24]:
sales_calender[sales_calender['availability'] == 0]

,unique_id,date,warehouse,total_orders,sales,sell_price_main,availability,holiday_name,holiday,shops_closed,winter_school_holidays,school_holidays,max_discount,is_discount


In [25]:
warehouse_rates = {
    'Prague_1': 1.0,
    'Prague_2': 1.0,
    'Prague_3': 1.0,
    'Brno_1': 1.0,
    'Budapest_1': 0.063,
    'Munich_1': 25.20,
    'Frankfurt_1': 25.20
}

In [26]:
sales_calender['actual_price_czk'] = (
    sales_calender['sell_price_main']
    * (1 - sales_calender['max_discount'])
    * sales_calender['warehouse'].map(warehouse_rates)
).round(2)

In [27]:
sales_calender['real_revenue_czk'] = (
    sales_calender['actual_price_czk']
    * sales_calender['sales']
    * sales['availability']
).round(2)

In [28]:
sales_calender["lost_revenue_czk"] = (
    sales_calender['actual_price_czk']
    * sales_calender['sales']
    * ((1 - sales_calender['availability']))
).round(2)

In [29]:
sales_calender['potential_revenue_czk'] = sales_calender['real_revenue_czk'] + sales_calender['lost_revenue_czk']

In [30]:
sales_calender['holiday_name'] = np.where(
    sales_calender['holiday_name'].notna(),
    sales_calender['holiday_name'],
    np.where(sales_calender['holiday'] == 1, 'Unnamed Holiday', 'No Holiday')
)

In [31]:
sales_calender[['total_orders', 'sales', 'real_revenue_czk', 'lost_revenue_czk', 'potential_revenue_czk']] = sales_calender[['total_orders', 'sales', 'real_revenue_czk', 'lost_revenue_czk', 'potential_revenue_czk']].fillna(0)

In [32]:
sales_calender[(sales_calender['sales'] == 0) & (sales_calender['holiday'] == 1)].groupby(['holiday', 'shops_closed']).size()

holiday  shops_closed
1        0               2880
         1               4546
dtype: int64

In [33]:
sales_calender.shape

(4007377, 18)

In [34]:
sales_calender[(sales_calender['real_revenue_czk'] == 0)]

,unique_id,date,warehouse,total_orders,sales,sell_price_main,availability,holiday_name,holiday,shops_closed,winter_school_holidays,school_holidays,max_discount,is_discount,actual_price_czk,real_revenue_czk,lost_revenue_czk,potential_revenue_czk
11,4845,2020-12-24,Budapest_1,2998.0,0.0,494.02,1.00,Christmas Eve,1,0,0,0,0.00000,0,31.12,0.0,0.0,0.0
82,4845,2022-04-16,Budapest_1,3481.0,0.0,486.41,1.00,Memorial Day for the Victims of the Holocaust,1,0,1,0,0.15649,1,25.85,0.0,0.0,0.0
152,4845,2020-10-13,Budapest_1,4144.0,0.0,494.02,1.00,No Holiday,0,0,0,0,0.00000,0,31.12,0.0,0.0,0.0
183,4845,2021-12-24,Budapest_1,3115.0,0.0,455.96,1.00,Christmas Eve,1,0,0,0,0.15025,1,24.41,0.0,0.0,0.0
355,4845,2021-05-29,Budapest_1,4250.0,0.0,455.96,1.00,No Holiday,0,0,0,0,0.15025,1,24.41,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4005936,5061,2021-07-15,Prague_2,4052.0,0.0,110.46,0.03,No Holiday,0,0,0,0,0.00000,0,110.46,0.0,0.0,0.0
4006454,1081,2023-09-26,Brno_1,8014.0,0.0,22.88,0.02,No Holiday,0,0,0,0,0.00000,0,22.88,0.0,0.0,0.0
4006646,1218,2023-06-23,Budapest_1,5916.0,0.0,1416.83,0.03,No Holiday,0,0,0,0,0.00000,0,89.26,0.0,0.0,0.0
4006647,1218,2023-06-20,Budapest_1,5641.0,0.0,1416.83,0.06,No Holiday,0,0,0,0,0.00000,0,89.26,0.0,0.0,0.0


## Data Quality Summary

- **Currency:** Czech Republic — CZK, Hungary — HUF, Germany — EUR
- **Missed values:** 42 out of 4 million — 0.00105%. Can be dropped these 42
- **Discount:** Chosen max one from all discount types
- **Availability:** Part of day when the good is available
- **Total_orders:** Days when unexpectedly more than 2 unique values of total_orders: 7517, so because of that I take usual 2 unique values per warehouse. Take them in load_clean.py to import them into sql
- **Outliers:** 3 outliers in Hungary warehouse. Assumptions that are higher price due to type of good (Meat/fish - Salmon)
- **Revenue:** Sales * actual_price_czk * availability
- **Potential_revenue:** Calculated with the help of (1 - availability)/availability as lost_revenue
- **Holiday_day:** 2880 shops_open during holiday, sales - 0 (nobody bought) / 4546 shops_closed during holiday, sales - 0 (shops are closed)
- **Sales - 0:** 41791 raws with 0 sales - maybe goods had been already sold or nobody bought them that day
- **Currency conversion:** Standardized all prices and revenue to CZK (`actual_price_czk`, `revenue_czk`) for cross-warehouse comparison using fixed exchange rates: 1 EUR = 25.20 CZK (Munich, Frankfurt) and 1 HUF = 0.063 CZK (1 CZK ≈ 15.87 HUF, Budapest).

In [35]:
inventory = pd.read_csv('../data/raw/inventory.csv')
inventory.head()

,unique_id,product_unique_id,name,L1_category_name_en,L2_category_name_en,L3_category_name_en,L4_category_name_en,warehouse
0,5255,2583,Pastry_196,Bakery,Bakery_L2_14,Bakery_L3_26,Bakery_L4_1,Prague_3
1,4948,2426,Herb_19,Fruit and vegetable,Fruit and vegetable_L2_30,Fruit and vegetable_L3_86,Fruit and vegetable_L4_1,Prague_3
2,2146,1079,Beet_2,Fruit and vegetable,Fruit and vegetable_L2_3,Fruit and vegetable_L3_65,Fruit and vegetable_L4_34,Prague_1
3,501,260,Chicken_13,Meat and fish,Meat and fish_L2_13,Meat and fish_L3_27,Meat and fish_L4_5,Prague_1
4,4461,2197,Chicory_1,Fruit and vegetable,Fruit and vegetable_L2_17,Fruit and vegetable_L3_33,Fruit and vegetable_L4_1,Frankfurt_1


In [36]:
inventory.isnull().sum()

unique_id              0
product_unique_id      0
name                   0
L1_category_name_en    0
L2_category_name_en    0
L3_category_name_en    0
L4_category_name_en    0
warehouse              0
dtype: int64

In [37]:
inventory.nunique()

unique_id              5432
product_unique_id      2670
name                   2670
L1_category_name_en       3
L2_category_name_en      47
L3_category_name_en     177
L4_category_name_en      68
warehouse                 7
dtype: int64

In [38]:
inventory['L1_category_name_en'].unique()

<ArrowStringArray>
['Bakery', 'Fruit and vegetable', 'Meat and fish']
Length: 3, dtype: str

In [39]:
merged_sales = pd.merge(sales_calender, inventory,  on = ['unique_id', 'warehouse'], how = 'left')
merged_sales

,unique_id,date,warehouse,total_orders,sales,sell_price_main,availability,holiday_name,holiday,shops_closed,...,actual_price_czk,real_revenue_czk,lost_revenue_czk,potential_revenue_czk,product_unique_id,name,L1_category_name_en,L2_category_name_en,L3_category_name_en,L4_category_name_en
0,4845,2024-03-10,Budapest_1,6436.0,16.34,646.26,1.00,No Holiday,0,0,...,34.48,563.40,0.00,563.40,2375,Croissant_35,Bakery,Bakery_L2_18,Bakery_L3_83,Bakery_L4_1
1,4845,2021-05-25,Budapest_1,4663.0,12.63,455.96,1.00,No Holiday,0,0,...,24.41,308.30,0.00,308.30,2375,Croissant_35,Bakery,Bakery_L2_18,Bakery_L3_83,Bakery_L4_1
2,4845,2021-12-20,Budapest_1,6507.0,34.55,455.96,1.00,No Holiday,0,0,...,24.41,843.37,0.00,843.37,2375,Croissant_35,Bakery,Bakery_L2_18,Bakery_L3_83,Bakery_L4_1
3,4845,2023-04-29,Budapest_1,5463.0,34.52,646.26,0.96,No Holiday,0,0,...,32.56,1079.01,44.96,1123.97,2375,Croissant_35,Bakery,Bakery_L2_18,Bakery_L3_83,Bakery_L4_1
4,4845,2022-04-01,Budapest_1,5997.0,35.92,486.41,1.00,No Holiday,0,0,...,25.85,928.53,0.00,928.53,2375,Croissant_35,Bakery,Bakery_L2_18,Bakery_L3_83,Bakery_L4_1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4007372,4941,2023-06-21,Prague_1,9988.0,26.56,34.06,1.00,No Holiday,0,0,...,34.06,904.63,0.00,904.63,2422,Kohlrabi_9,Fruit and vegetable,Fruit and vegetable_L2_3,Fruit and vegetable_L3_114,Fruit and vegetable_L4_1
4007373,4941,2023-06-24,Prague_1,8518.0,27.42,34.06,1.00,No Holiday,0,0,...,34.06,933.93,0.00,933.93,2422,Kohlrabi_9,Fruit and vegetable,Fruit and vegetable_L2_3,Fruit and vegetable_L3_114,Fruit and vegetable_L4_1
4007374,4941,2023-06-23,Prague_1,10424.0,33.39,34.06,1.00,No Holiday,0,0,...,34.06,1137.26,0.00,1137.26,2422,Kohlrabi_9,Fruit and vegetable,Fruit and vegetable_L2_3,Fruit and vegetable_L3_114,Fruit and vegetable_L4_1
4007375,4941,2023-06-22,Prague_1,10342.0,22.88,34.06,1.00,No Holiday,0,0,...,34.06,779.29,0.00,779.29,2422,Kohlrabi_9,Fruit and vegetable,Fruit and vegetable_L2_3,Fruit and vegetable_L3_114,Fruit and vegetable_L4_1


In [40]:
merged_sales.isnull().sum()

unique_id                 0
date                      0
warehouse                 0
total_orders              0
sales                     0
sell_price_main           0
availability              0
holiday_name              0
holiday                   0
shops_closed              0
winter_school_holidays    0
school_holidays           0
max_discount              0
is_discount               0
actual_price_czk          0
real_revenue_czk          0
lost_revenue_czk          0
potential_revenue_czk     0
product_unique_id         0
name                      0
L1_category_name_en       0
L2_category_name_en       0
L3_category_name_en       0
L4_category_name_en       0
dtype: int64

In [41]:
def clean_columns(df):
    df = df.copy()
    df.columns = (
        df.columns.str.strip()
        .str.lower()
        .str.replace(r"\s+", "_", regex=True)
    )
    return df

In [42]:
clean_columns(merged_sales)

,unique_id,date,warehouse,total_orders,sales,sell_price_main,availability,holiday_name,holiday,shops_closed,...,actual_price_czk,real_revenue_czk,lost_revenue_czk,potential_revenue_czk,product_unique_id,name,l1_category_name_en,l2_category_name_en,l3_category_name_en,l4_category_name_en
0,4845,2024-03-10,Budapest_1,6436.0,16.34,646.26,1.00,No Holiday,0,0,...,34.48,563.40,0.00,563.40,2375,Croissant_35,Bakery,Bakery_L2_18,Bakery_L3_83,Bakery_L4_1
1,4845,2021-05-25,Budapest_1,4663.0,12.63,455.96,1.00,No Holiday,0,0,...,24.41,308.30,0.00,308.30,2375,Croissant_35,Bakery,Bakery_L2_18,Bakery_L3_83,Bakery_L4_1
2,4845,2021-12-20,Budapest_1,6507.0,34.55,455.96,1.00,No Holiday,0,0,...,24.41,843.37,0.00,843.37,2375,Croissant_35,Bakery,Bakery_L2_18,Bakery_L3_83,Bakery_L4_1
3,4845,2023-04-29,Budapest_1,5463.0,34.52,646.26,0.96,No Holiday,0,0,...,32.56,1079.01,44.96,1123.97,2375,Croissant_35,Bakery,Bakery_L2_18,Bakery_L3_83,Bakery_L4_1
4,4845,2022-04-01,Budapest_1,5997.0,35.92,486.41,1.00,No Holiday,0,0,...,25.85,928.53,0.00,928.53,2375,Croissant_35,Bakery,Bakery_L2_18,Bakery_L3_83,Bakery_L4_1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4007372,4941,2023-06-21,Prague_1,9988.0,26.56,34.06,1.00,No Holiday,0,0,...,34.06,904.63,0.00,904.63,2422,Kohlrabi_9,Fruit and vegetable,Fruit and vegetable_L2_3,Fruit and vegetable_L3_114,Fruit and vegetable_L4_1
4007373,4941,2023-06-24,Prague_1,8518.0,27.42,34.06,1.00,No Holiday,0,0,...,34.06,933.93,0.00,933.93,2422,Kohlrabi_9,Fruit and vegetable,Fruit and vegetable_L2_3,Fruit and vegetable_L3_114,Fruit and vegetable_L4_1
4007374,4941,2023-06-23,Prague_1,10424.0,33.39,34.06,1.00,No Holiday,0,0,...,34.06,1137.26,0.00,1137.26,2422,Kohlrabi_9,Fruit and vegetable,Fruit and vegetable_L2_3,Fruit and vegetable_L3_114,Fruit and vegetable_L4_1
4007375,4941,2023-06-22,Prague_1,10342.0,22.88,34.06,1.00,No Holiday,0,0,...,34.06,779.29,0.00,779.29,2422,Kohlrabi_9,Fruit and vegetable,Fruit and vegetable_L2_3,Fruit and vegetable_L3_114,Fruit and vegetable_L4_1


In [43]:
merged_sales = merged_sales.drop(columns=['winter_school_holidays', 'school_holidays'])

In [44]:
merged_sales.to_parquet('../data/analyzed/rohlik_cleaned.parquet', index=False)